# 01 — Python & Data Foundations
Put `appointments.csv` and `clinics.csv` beside this notebook. Run top-to-bottom to generate all Part 1 outputs.

In [ ]:
import csv, json, requests
import numpy as np
import pandas as pd
from pathlib import Path
APPOINTMENTS_PATH=Path("appointments.csv")
CLINICS_PATH=Path("clinics.csv")

## Task 1.1 — CSV module ingestion (built-in csv/json only)

In [ ]:
accepted,rejected=[],[]
def optional_int(value):
    if not value.strip():
        return None
    numeric=float(value)
    if not numeric.is_integer():
        raise ValueError(f"Expected a whole number, received {value!r}")
    return int(numeric)
def optional_float(value):
    return float(value) if value.strip() else None

with open(APPOINTMENTS_PATH,newline="",encoding="utf-8") as f:
    reader=csv.DictReader(f)
    headers=reader.fieldnames
    for row in reader:
        try:
            if not row["age"].strip():
              raise ValueError("Missing age")

            if not row["distance_km"].strip():
             raise ValueError("Missing distance_km")

            row["age"]=optional_int(row["age"])
            row["fee_inr"]=optional_int(row["fee_inr"])
            row["prior_no_shows"]=optional_int(row["prior_no_shows"])
            row["distance_km"]=optional_float(row["distance_km"])
            accepted.append(row)
        except ValueError as exc:
            rejected.append({"row":row,"error":str(exc)})
print("Header:",headers)
print("Rows:",len(accepted)+len(rejected),"Accepted:",len(accepted),"Rejected malformed rows:",len(rejected))
print("Missing numeric values retained for pandas cleaning:",
      sum(r["age"] is None for r in accepted),"age;",
      sum(r["distance_km"] is None for r in accepted),"distance_km")
with open("rejected_rows.json","w",encoding="utf-8") as f:
    json.dump(rejected,f,indent=2)


Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Rows: 1020 Accepted: 988 Rejected malformed rows: 32
Missing numeric values retained for pandas cleaning: 0 age; 0 distance_km


## Task 1.2 — Pandas cleaning pass 1

In [ ]:
df=pd.read_csv(APPOINTMENTS_PATH)
print("Shape:",df.shape); df.info()
print("Nulls:\\n",df.isnull().sum())
print("Duplicates:",df.duplicated().sum())
df=df.drop_duplicates()
print("After duplicate removal:",len(df))
print("Raw distinct channels:",
      df["booking_channel"].nunique())

print("Raw channel values:",
      sorted(df["booking_channel"].dropna().unique()))
df["booking_channel"] = (
    df["booking_channel"]
    .str.strip()
    .str.lower()
    .replace({
        "app": "App",
        "mobile app": "App",
        "phone": "Phone",
        "call": "Phone",
        "walk-in": "Walk-In",
        "walk in": "Walk-In",
        "walkin": "Walk-In"
    })
)
print("Channels:",sorted(df["booking_channel"].dropna().unique()))
df["appt_date"]=pd.to_datetime(df["appt_date"],errors="coerce")
df["booked_on"]=pd.to_datetime(df["booked_on"])
df=df.dropna(subset=["appt_date","status"]).copy()
print("After missing appt_date/status removal:",len(df))

Shape: (1020, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB
Nulls:\n appointment_id      0
patient_id          0
clinic_id           0

## Task 1.3 — Validity, imputation, and outliers

In [ ]:

# ============================================
# TASK 1.3: DATA CLEANING & OUTLIER TREATMENT
# ============================================

import numpy as np

# Step 1: Verify cleaned dataset
print("Initial Shape:", df.shape)

assert len(df) == 990, (
    "Please run Task 1.2 again. Expected 990 rows."
)

# Step 2: Identify invalid ages
invalid_age_mask = (df["age"] < 1) | (df["age"] > 100)

invalid_age_count = invalid_age_mask.sum()

# Replace invalid ages with NaN
df.loc[invalid_age_mask, "age"] = np.nan

print("\nInvalid ages:", invalid_age_count)
print("Missing ages after validity:", df["age"].isna().sum())


# Step 3: Calculate median values
age_median = df["age"].median()
distance_median = df["distance_km"].median()

print("\nAge median:", age_median)
print("Distance median:", distance_median)


# Step 4: Impute missing values

df["age"] = (
    df["age"]
    .fillna(age_median)
    .round()
    .astype(int)
)

df["distance_km"] = df["distance_km"].fillna(
    distance_median
)


# Step 5: Detect and cap outliers using IQR

for col in ["distance_km", "fee_inr"]:

    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)

    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_count = (
        (df[col] < lower_bound) |
        (df[col] > upper_bound)
    ).sum()

    print(f"\nColumn: {col}")
    print("Q1:", q1)
    print("Q3:", q3)
    print("IQR:", iqr)
    print("Lower Bound:", lower_bound)
    print("Upper Bound:", upper_bound)
    print("Outliers capped:", outlier_count)

    df[col] = np.clip(
        df[col],
        lower_bound,
        upper_bound
    )


# Step 6: Final rounding

df["distance_km"] = df["distance_km"].round(1)

df["fee_inr"] = (
    df["fee_inr"]
    .round()
    .astype(int)
)


# Step 7: Final validation

print("\n========== TASK 1.3 FINAL VALIDATION ==========")

print("Final Shape:", df.shape)

print("Maximum Age:", df["age"].max())
print("Minimum Age:", df["age"].min())

print("Maximum Fee:", df["fee_inr"].max())

print("Missing Values Across All Columns:",
      df.isna().sum().sum())




Initial Shape: (990, 14)

Invalid ages: 5
Missing ages after validity: 17

Age median: 48.0
Distance median: 8.25

Column: distance_km
Q1: 4.625
Q3: 13.0
IQR: 8.375
Lower Bound: -7.9375
Upper Bound: 25.5625
Outliers capped: 26

Column: fee_inr
Q1: 500.0
Q3: 787.5
IQR: 287.5
Lower Bound: 68.75
Upper Bound: 1218.75
Outliers capped: 3

========== TASK 1.3 FINAL VALIDATION ==========
Final Shape: (990, 14)
Maximum Age: 78
Minimum Age: 18
Maximum Fee: 1219
Missing Values Across All Columns: 0

Task 1.3 completed!


## Task 1.4 — Features and vectorized NumPy

In [ ]:
df["lead_days"]=(df.appt_date-df.booked_on).dt.days
df["no_show"]=(df.status=="No-Show").astype(int)
X_np=df[["age","distance_km","fee_inr","lead_days"]].to_numpy()
print("Column means:",X_np.mean(axis=0))
long_mask=df.lead_days.to_numpy()>14
print("lead_days >14:",long_mask.sum(),"share (%):",round(100*long_mask.mean(),1))
dist=df.distance_km.to_numpy()
z=(dist-np.mean(dist))/np.std(dist)
print("|distance z| >2:",np.sum(np.abs(z)>2))
print("No-shows:",df.no_show.sum(),"share (%):",round(100*df.no_show.mean(),1))

Column means: [ 48.17373737   9.50090909 635.71414141  11.81919192]
lead_days >14: 296 share (%): 29.9
|distance z| >2: 53
No-shows: 218 share (%): 22.0


## Task 1.5 — Live Open-Meteo weather enrichment

In [ ]:

clinics = pd.read_csv(CLINICS_PATH)

def fetch_temp(row):
    params = {
        "latitude": row.latitude,
        "longitude": row.longitude,
        "hourly": "temperature_2m",
        "forecast_days": 1
    }

    reason = "unknown error"

    for attempt in range(2):
        try:
            response = requests.get(
                "https://api.open-meteo.com/v1/forecast",
                params=params,
                timeout=10
            )

            if response.status_code != 200:
                reason = f"HTTP status {response.status_code}"
                continue

            payload = response.json()

            temps = payload.get("hourly", {}).get("temperature_2m")

            if not temps or temps[0] is None:
                reason = "Missing hourly temperature"
                continue

            return temps[0]

        except requests.exceptions.RequestException as exc:
            reason = str(exc)

        except (ValueError, KeyError, TypeError) as exc:
            reason = f"Invalid response: {exc}"

    print(f"{row.clinic_name}: {reason}")
    return None


clinics["temp_c"] = clinics.apply(fetch_temp, axis=1)

clinics.to_csv("clinics_enriched.csv", index=False)

print(clinics[["clinic_name", "temp_c"]])


                clinic_name  temp_c
0     Kothrud Family Clinic    14.4
1      Hadapsar Care Centre    15.7
2    Sitabuldi Health Point    22.3
3      Malviya Nagar Clinic    15.3
4     Vaishali Nagar Clinic    15.3
5  Edappally Medical Centre    24.2
6        Vijay Nagar Clinic    15.2
7    Gomti Nagar Care Point    17.0


## Task 1.6 — Export, assertions, SQL generation, clinic groupby verification

In [ ]:
df=df.sort_values("appointment_id").copy()
for col in ["booked_on","appt_date"]:
    df[col]=pd.to_datetime(df[col])
    df[col]=df[col].dt.strftime("%Y-%m-%d")
cols=["appointment_id","patient_id","clinic_id","booked_on","appt_date","age","gender",
      "distance_km","fee_inr","booking_channel","visit_type","reminder_sent",
      "prior_no_shows","status","lead_days","no_show"]
df=df[cols]
assert df.appointment_id.nunique()==len(df)
assert df.appointment_id.isna().sum()==0
assert df.isna().sum().sum()==0, "Expected zero nulls across all 16 exported columns"
print("Unique IDs:",df.appointment_id.nunique(),"Null appointment IDs:",df.appointment_id.isna().sum(),"Nulls across all columns:",int(df.isna().sum().sum()))
df.to_csv("appointments_clean.csv",index=False)

joined=df.merge(clinics[["clinic_id","clinic_name","city"]],on="clinic_id",how="left")
summary=(joined.groupby(["clinic_name","city"])
         .agg(appointments=("appointment_id","count"),no_shows=("no_show","sum")).reset_index())
summary["no_show_pct"]=(100*summary.no_shows/summary.appointments).round(1)
print(summary.sort_values("no_show_pct",ascending=False).to_string(index=False))

def sql_value(v):
    if pd.isna(v): return "NULL"
    if isinstance(v,(int,np.integer)): return str(int(v))
    if isinstance(v,(float,np.floating)): return str(float(v))
    return "'"+str(v).replace("\\","\\\\").replace("'","''")+"'"
with open("load_data.sql","w",encoding="utf-8") as f:
    f.write("USE noshowguard;\n")
    clinic_cols=["clinic_id","clinic_name","city","tier","latitude","longitude","temp_c"]
    for _,r in clinics[clinic_cols].iterrows():
        f.write("INSERT INTO clinics VALUES ("+", ".join(sql_value(r[c]) for c in clinic_cols)+");\n")
    for _,r in df.iterrows():
        f.write("INSERT INTO appointments VALUES ("+", ".join(sql_value(r[c]) for c in cols)+");\n")
print("INSERT statements:",len(clinics)+len(df))

Unique IDs: 990 Null appointment IDs: 0 Nulls across all columns: 0
             clinic_name    city  appointments  no_shows  no_show_pct
  Gomti Nagar Care Point Lucknow            90        30         33.3
    Hadapsar Care Centre    Pune           143        39         27.3
    Malviya Nagar Clinic  Jaipur           142        38         26.8
   Kothrud Family Clinic    Pune           160        36         22.5
      Vijay Nagar Clinic  Indore           101        21         20.8
Edappally Medical Centre   Kochi           142        23         16.2
  Sitabuldi Health Point  Nagpur            82        13         15.9
   Vaishali Nagar Clinic  Jaipur           130        18         13.8
INSERT statements: 998
